# PakRoad FloodWatch — Deliverable 3: Predictive ML Model vs Declared Baseline
**HackInsight 2026: Climate & Infrastructure Track**  
This notebook implements the complete evaluation pipeline matching HackInsight Guide Chapter 8:
1. **Honest Time-Based Split:** Train on 2021–2024 (403k rows), Test on 2025–2026 (188k rows)
2. **Declared Baseline Evaluation:** 24h persistence model (̂_t = y_{t-24h}$)
3. **Primary LightGBM Regressor:** Point forecast with $\ge 20\%$ MAE improvement over baseline
4. **Uncertainty Quantification:** Multi-quantile prediction intervals ({10}, Q_{50}, Q_{90}$)
5. **LightGBM Classifier:** Road inundation risk classification & Brier score
6. **Group Fairness Analysis:** Disaggregated audit across all 12 cities, provinces, and seasons

In [ ]:
# Step 1: Install required dependencies
!pip install -q lightgbm scikit-learn joblib matplotlib seaborn pandas pyarrow

In [ ]:
# Step 2: Load the full 592k-row weather dataset
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

DATA_PATH = "data/processed/weather_hourly.parquet"
if not os.path.exists(DATA_PATH):
    # Check alternate paths
    alt = "weather_hourly.parquet"
    if os.path.exists(alt):
        DATA_PATH = alt
    else:
        raise FileNotFoundError(f"Cannot find {DATA_PATH}. Please ensure weather_hourly.parquet is present.")

print(f"Loading dataset from {DATA_PATH}...")
df = pd.read_parquet(DATA_PATH)
print(f"Successfully loaded {len(df):,} hourly records across {df['city'].nunique()} cities.")
df.head(3)

In [ ]:
# Step 3: Feature Engineering & Temporal Lags
CITY_PROVINCE_MAP = {
    "Karachi": "Sindh", "Hyderabad": "Sindh", "Sukkur": "Sindh",
    "Lahore": "Punjab", "Faisalabad": "Punjab", "Multan": "Punjab", "Rawalpindi": "Punjab",
    "Islamabad": "Islamabad Capital Territory", "Peshawar": "Khyber Pakhtunkhwa",
    "Quetta": "Balochistan", "Gwadar": "Balochistan", "Gilgit": "Gilgit-Baltistan"
}

time_col = "timestamp_pkt" if "timestamp_pkt" in df.columns else "timestamp_utc"
df[time_col] = pd.to_datetime(df[time_col])
df["year"] = df[time_col].dt.year
df["month"] = df[time_col].dt.month
df["day"] = df[time_col].dt.day
df["hour"] = df[time_col].dt.hour
df["province"] = df["city"].map(CITY_PROVINCE_MAP).fillna("Other")

# Rename aliases
rename_map = {
    "precipitation_mm": "precipitation",
    "rain_mm": "rain",
    "temperature_2m_c": "temperature_2m",
    "relative_humidity_pct": "relative_humidity_2m",
    "soil_moisture_0_to_7cm_m3m3": "soil_moisture_0_7cm",
    "wind_speed_10m_kmh": "wind_speed_10m",
    "weather_code_wmo": "weather_code",
}
for k, v in rename_map.items():
    if k in df.columns and v not in df.columns:
        df[v] = df[k]

df["rain"] = df["rain"].fillna(0.0)
df["precipitation"] = df["precipitation"].fillna(df["rain"])

# Per-city lag and rolling features
df = df.sort_values(["city", time_col]).reset_index(drop=True)
city_grp = df.groupby("city")
df["rain_lag_1h"] = city_grp["rain"].shift(1).fillna(0.0)
df["rain_lag_2h"] = city_grp["rain"].shift(2).fillna(0.0)
df["rain_lag_24h"] = city_grp["rain"].shift(24).fillna(0.0) # DECLARED 24h PERSISTENCE BASELINE
df["rain_rolling_3h_sum"] = city_grp["rain"].rolling(3, min_periods=1).sum().reset_index(0, drop=True).fillna(0.0)
df["rain_rolling_6h_sum"] = city_grp["rain"].rolling(6, min_periods=1).sum().reset_index(0, drop=True).fillna(0.0)

if "surface_pressure_hpa" in df.columns:
    df["pressure_trend_3h"] = city_grp["surface_pressure_hpa"].diff(3).fillna(0.0)
else:
    df["pressure_trend_3h"] = 0.0

# Cyclic time & monsoon indicators
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)
df["is_monsoon"] = df["month"].isin([7, 8, 9]).astype(int)

# Atmospheric & saturation interaction
dew_approx = df["temperature_2m"] - ((100 - df["relative_humidity_2m"]) / 5.0)
df["vapor_saturation_gap"] = df["temperature_2m"] - dew_approx
df["soil_rain_saturation_index"] = df["soil_moisture_0_7cm"].fillna(0) * (df["rain_rolling_6h_sum"] + 0.1)
df["convective_heat_trigger"] = ((df["temperature_2m"] >= 35.0) & (df["relative_humidity_2m"] >= 65.0)).astype(int)

# Road condition hazard proxy
water_proxy = (df["rain_rolling_3h_sum"] * 0.5) + (df["rain"] * 1.0) + (df["soil_moisture_0_7cm"].fillna(0) * 10.0)
df["road_condition_code"] = 0
df.loc[water_proxy >= 5.0, "road_condition_code"] = 1
df.loc[water_proxy >= 15.0, "road_condition_code"] = 2

print("Feature engineering complete! Shape:", df.shape)

In [ ]:
# Step 4: Honest Time-Based Train/Test Split
train_mask = df["year"] <= 2024
test_mask = df["year"] >= 2025

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

print(f"Train observations (2021-2024): {len(train_df):,} rows")
print(f"Test observations  (2025-2026): {len(test_df):,} rows (Untouched Future Set)")

cities = sorted(train_df["city"].unique())
city_to_code = {c: i for i, c in enumerate(cities)}
train_df["city_code"] = train_df["city"].map(city_to_code).fillna(-1)
test_df["city_code"] = test_df["city"].map(city_to_code).fillna(-1)

feature_cols = [
    "rain_lag_1h", "rain_lag_2h", "rain_lag_24h",
    "rain_rolling_3h_sum", "rain_rolling_6h_sum", "pressure_trend_3h",
    "soil_moisture_0_7cm", "relative_humidity_2m", "temperature_2m",
    "wind_speed_10m", "weather_code", "hour_sin", "hour_cos",
    "month_sin", "month_cos", "is_monsoon", "vapor_saturation_gap",
    "soil_rain_saturation_index", "convective_heat_trigger", "city_code"
]

X_train = train_df[feature_cols]
y_train = train_df["precipitation"]
X_test = test_df[feature_cols]
y_test = test_df["precipitation"]

y_train_cls = (train_df["road_condition_code"] >= 1).astype(int)
y_test_cls = (test_df["road_condition_code"] >= 1).astype(int)

In [ ]:
# Step 5: Evaluate Declared 24h Persistence Baseline vs LightGBM Model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, brier_score_loss
import lightgbm as lgb

# 1. Declared Baseline Evaluation
y_baseline_pred = test_df["rain_lag_24h"].values
baseline_mae = mean_absolute_error(y_test, y_baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_baseline_pred))
baseline_brier = brier_score_loss(y_test_cls, (y_baseline_pred >= 2.5).astype(int))

print(f"Declared 24h Baseline MAE:  {baseline_mae:.4f} mm")
print(f"Declared 24h Baseline RMSE: {baseline_rmse:.4f} mm")

# 2. Train Primary LightGBM Regressor
print("
Training Primary LightGBM Regressor...")
lgb_reg = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
lgb_reg.fit(X_train, y_train)
y_model_pred = np.clip(lgb_reg.predict(X_test), 0.0, None)

model_mae = mean_absolute_error(y_test, y_model_pred)
model_rmse = np.sqrt(mean_squared_error(y_test, y_model_pred))
model_r2 = r2_score(y_test, y_model_pred)

mae_gain = ((baseline_mae - model_mae) / baseline_mae) * 100.0
rmse_gain = ((baseline_rmse - model_rmse) / baseline_rmse) * 100.0

print(f"LightGBM Regressor MAE:     {model_mae:.4f} mm")
print(f"LightGBM Regressor RMSE:    {model_rmse:.4f} mm")
print(f"Explanatory Power (R²):     {model_r2:.4f}")
print(f"
===> Relative MAE Improvement:  {mae_gain:+.2f}%")
print(f"===> Relative RMSE Improvement: {rmse_gain:+.2f}%")
if mae_gain >= 20.0:
    print("
[PASSED] ✅ Beated declared baseline by >= 20% on future test set!")
else:
    print("
[NOTE] Gain is under 20% milestone.")

In [ ]:
# Step 6: Multi-Quantile Uncertainty Intervals (Q10, Q50, Q90)
print("Training Multi-Quantile LightGBM models for uncertainty intervals...")
q_preds = {}
for q in [0.10, 0.50, 0.90]:
    qm = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=150, learning_rate=0.08, num_leaves=25, random_state=42, n_jobs=-1)
    qm.fit(X_train, y_train)
    q_preds[q] = np.clip(qm.predict(X_test), 0.0, None)

q10, q50, q90 = q_preds[0.10], q_preds[0.50], q_preds[0.90]
coverage = np.mean((y_test.values >= q10) & (y_test.values <= q90)) * 100.0
avg_width = np.mean(q90 - q10)
print(f"Empirical 80% Prediction Interval Coverage: {coverage:.2f}%")
print(f"Average Uncertainty Interval Width: {avg_width:.2f} mm")

In [ ]:
# Step 7: Road Inundation Classification & Brier Score
lgb_cls = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.05, num_leaves=31, class_weight="balanced", random_state=42, n_jobs=-1)
lgb_cls.fit(X_train, y_train_cls)
cls_probs = lgb_cls.predict_proba(X_test)[:, 1]
model_brier = brier_score_loss(y_test_cls, cls_probs)
brier_gain = ((baseline_brier - model_brier) / baseline_brier) * 100.0
print(f"Baseline Brier Score: {baseline_brier:.4f}")
print(f"LightGBM Classifier Brier Score: {model_brier:.4f}")
print(f"Brier Score Calibration Gain: {brier_gain:+.2f}%")

In [ ]:
# Step 8: Group Fairness Analysis by City, Province, and Season
test_eval = test_df.copy()
test_eval["actual"] = y_test.values
test_eval["err_base"] = np.abs(test_eval["actual"] - y_baseline_pred)
test_eval["err_model"] = np.abs(test_eval["actual"] - y_model_pred)

city_audit = test_eval.groupby("city").agg(
    records=("actual", "count"),
    baseline_mae=("err_base", "mean"),
    model_mae=("err_model", "mean")
).reset_index()
city_audit["gain_pct"] = ((city_audit["baseline_mae"] - city_audit["model_mae"]) / city_audit["baseline_mae"]) * 100.0
city_audit = city_audit.sort_values(by="model_mae", ascending=False)

print("
--- Disaggregated Fairness Table (City Breakdown) ---")
display(city_audit)

weakest = city_audit.iloc[0]
print(f"
[AUDIT] Weakest City: {weakest['city']} (MAE: {weakest['model_mae']:.4f} mm)")
print(f"Actionable Operational Guidance: Municipal dispatchers for {weakest['city']} must integrate upstream streamflow gauges before lifting road hazard alerts.")

In [ ]:
# Step 9: Visualizations & Saving Model Binaries
import joblib
os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

joblib.dump(lgb_reg, "models/lightgbm_rainfall_regressor.joblib")
joblib.dump(lgb_cls, "models/lightgbm_inundation_classifier.joblib")
print("Saved models to models/")

# Plot 1: Baseline vs Model MAE
plt.figure(figsize=(6, 4))
bars = plt.bar(["24h Persistence Baseline", "LightGBM Model"], [baseline_mae, model_mae], color=["#94a3b8", "#2563eb"], width=0.5)
plt.title("Rainfall Forecast MAE: Baseline vs LightGBM (2025–2026 Test Set)", fontweight="bold")
plt.ylabel("MAE (mm)")
for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, f"{b.get_height():.4f} mm", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/baseline_vs_model_mae.png", dpi=150)
plt.show()

# Plot 2: Fairness Gain by City
plt.figure(figsize=(10, 4.5))
sns.barplot(data=city_audit, x="city", y="gain_pct", palette="Blues_r")
plt.axhline(20.0, color="#dc2626", linestyle="--", label="20% Improvement Target")
plt.title("Relative MAE Improvement by City (Group Fairness Analysis)", fontweight="bold")
plt.ylabel("Gain over Baseline (%)")
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/fairness_by_city.png", dpi=150)
plt.show()

print("All evaluation tasks and visualizations completed successfully!")